# Credit-default model comparison

Portfolio adaptation derived from Benjamin Buckland's AAI6610 Module 8 analysis. See ../README.md for attribution, AI assistance, dependency pins and evaluation limits. The original code and experiment sequence are retained, not represented as a new independent experiment.

The holdout was previously examined and informed subsequent model selection; results are exploratory, not an untouched unbiased final assessment. Neural architecture selection precedes its CV and categorical vocabulary spans some CV assessment rows. See README for details.


## Step 1: Import libraries

Same imports as Module 5, plus `f1_score` and `classification_report`.

In [ ]:
import time
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

from zipfile import ZipFile
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, f1_score, classification_report

print("TensorFlow version:", tf.__version__)

## Step 2: Load and clean the data

Same loading and cleaning as Modules 3 and 5: undocumented education and marital-status codes are labeled "Unknown", and the ID column is not used as a predictor.

In [ ]:
import os
import hashlib
from pathlib import Path

data_zip = Path(os.environ["CREDIT_DEFAULT_DATA_ZIP"]).expanduser()
expected_sha256 = "56c885f84457f6680f8438f02bfcdac9579323d8a94465ee5f26e32baa727602"
if hashlib.sha256(data_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError("Unexpected dataset checksum; verify official provenance")

with ZipFile(data_zip, "r") as zip_file:
    with zip_file.open("default of credit card clients.xls") as excel_file:
        df = pd.read_excel(excel_file, header=1, engine="xlrd")

df_clean = df.copy()

education_mapping = {
    0: "Unknown",
    1: "Graduate school",
    2: "University",
    3: "High school",
    4: "Other",
    5: "Unknown",
    6: "Unknown"
}

marriage_mapping = {
    0: "Unknown",
    1: "Married",
    2: "Single",
    3: "Other"
}

df_clean["EDUCATION"] = df_clean["EDUCATION"].map(education_mapping)
df_clean["MARRIAGE"] = df_clean["MARRIAGE"].map(marriage_mapping)

target = "default payment next month"

print("Rows and columns:", df_clean.shape)

## Step 3: Recreate the train/test split

Same arguments as Modules 3 and 5 (`test_size=0.20`, `random_state=20`, `stratify=y`), giving the same 24,000 training and 6,000 test clients. The check confirms that the test set begins with the same clients as in Module 5.

In [ ]:
X = df_clean.drop(columns=["ID", target])
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=20,
    stratify=y
)

print("Training predictors:", X_train.shape)
print("Test predictors:", X_test.shape)

# Same first five test clients as Module 5, so the split matches the midterm
assert list(X_test.index[:5]) == [3361, 15271, 15665, 3892, 20106], "The split differs from the midterm"
print("Split matches the midterm")

## Step 4: Define the preprocessing

One-hot encoding for the 9 categorical columns and standardization for the 14 numerical columns, as in Module 3. Each statistical model gets its own copy of this preprocessor inside a pipeline, so it is refitted on the training data of every cross-validation fold.

In [ ]:
categorical_columns = [
    "SEX", "EDUCATION", "MARRIAGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"
]

# All remaining predictors are numerical
numerical_columns = [
    column for column in X_train.columns
    if column not in categorical_columns
]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         categorical_columns),

        ("numerical", StandardScaler(),
         numerical_columns)
    ]
)

print("Categorical columns:", len(categorical_columns))
print("Numerical columns:", len(numerical_columns))

# Part A: Statistical models

## Step 5: Cross-validation with precision, recall, and F1

The models are evaluated with the same five-fold stratified cross-validation as Module 3. The unrestricted decision tree is included because its overfitting is what led to the depth limit. F1, the harmonic mean of precision and recall, is added alongside accuracy, precision, and recall, and training F1 is kept so that it can be compared with validation F1 as a check for overfitting.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=20
)

# The Module 3 pipelines; each model gets its own copy of the preprocessor
cv_pipelines = {
    "Logistic regression": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Decision tree (unrestricted)": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", DecisionTreeClassifier(random_state=20))
    ]),
    "Decision tree (depth 5)": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", DecisionTreeClassifier(max_depth=5, random_state=20))
    ]),
    "Random forest": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=20, n_jobs=-1))
    ])
}

In [ ]:
# The Module 3 metrics, plus F1
scoring = ["accuracy", "precision", "recall", "f1"]

cv_raw_results = {}

for model_name, pipeline in cv_pipelines.items():
    cv_raw_results[model_name] = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        error_score="raise"
    )
    print(f"{model_name}: done")

# The keys cross_validate() created for one model
print(sorted(cv_raw_results["Logistic regression"].keys()))

In [ ]:
cv_fold_results = {}

for model_name, cv_results in cv_raw_results.items():
    fold_results = pd.DataFrame({
        "Training accuracy": cv_results["train_accuracy"],
        "Validation accuracy": cv_results["test_accuracy"],
        "Validation precision": cv_results["test_precision"],
        "Validation recall": cv_results["test_recall"],
        # Training and validation F1 for each fold
        "Training F1": cv_results["train_f1"],
        "Validation F1": cv_results["test_f1"]
    }, index=range(1, 6))

    fold_results.index.name = "Fold"
    cv_fold_results[model_name] = fold_results

# Fold-by-fold results for one model, as in Module 3
cv_fold_results["Logistic regression"].round(4)

Means across the five folds. The accuracy, precision, and recall means match Module 3 (Table 1 of the midterm). The F1 gap is training F1 minus validation F1.

In [ ]:
cv_summary = pd.DataFrame({
    model_name: fold_results.mean()
    for model_name, fold_results in cv_fold_results.items()
}).T

cv_summary["F1 gap"] = cv_summary["Training F1"] - cv_summary["Validation F1"]

cv_summary.round(4)

Standard deviations across the five folds describe fold-to-fold score variability. They do not establish statistical significance of differences between models; folds overlap in their training data, and a formal comparison needs an appropriate paired evaluation design.

In [ ]:
cv_std = pd.DataFrame({
    model_name: fold_results.std()
    for model_name, fold_results in cv_fold_results.items()
}).T

cv_std.round(4)

# Part B: Neural network

## Step 6: Validation set and preprocessing for the network

As in Module 5, 20% of the training data (4,800 clients) is held out for early stopping and for comparing architectures. The network's preprocessor is fitted on the remaining 19,200 clients and produces dense arrays, which Keras expects.

In [ ]:
X_train_nn, X_valid, y_train_nn, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=20,
    stratify=y_train
)

nn_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_columns),

        ("numerical", StandardScaler(),
         numerical_columns)
    ]
)

X_train_nn_prepared = nn_preprocessor.fit_transform(X_train_nn)
X_valid_prepared = nn_preprocessor.transform(X_valid)
X_test_prepared = nn_preprocessor.transform(X_test)

print("Network training:", X_train_nn_prepared.shape)
print("Validation:", X_valid_prepared.shape)
print("Test:", X_test_prepared.shape)

## Step 7: Define the network

The same `build_model()` function as Module 5. Seeds and deterministic operations make training repeatable, but exact replication across platforms and package versions is not guaranteed.

In [ ]:
tf.config.experimental.enable_op_determinism()

def build_model(n_hidden=2, n_neurons=64, dropout_rate=0.0, learning_rate=1e-3, seed=20):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=X_train_nn_prepared.shape[1:]))
    for layer in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons, activation="relu",
                                        kernel_initializer="he_normal"))
        # Randomly switch off a fraction of neurons during training only
        if dropout_rate > 0:
            model.add(tf.keras.layers.Dropout(rate=dropout_rate))
    model.add(tf.keras.layers.Dense(1, activation="sigmoid"))
    model.compile(loss="binary_crossentropy",
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=["accuracy"])
    return model

## Step 8: A function for accuracy, precision, recall, and F1

`cross_validate()` calculated the metrics for the statistical models. The network is not a scikit-learn model, so `cross_validate()` cannot score it. Its metrics are calculated directly from its 0/1 predictions instead, on the validation set in Step 9 and on the five folds in Step 10. The same function is used for all four models on the test set in Step 12.

In [ ]:
def classification_metrics(y_true, y_pred):
    """Accuracy, plus precision, recall and F1 for the default class (1), from 0/1 predictions."""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred)
    }

In [ ]:
# A small example that can be checked by hand:
# 2 true positives, 1 false positive, 2 false negatives, 1 true negative
example_actual = [1, 1, 1, 1, 0, 0]
example_predicted = [1, 1, 0, 0, 1, 0]

example_metrics = classification_metrics(example_actual, example_predicted)
print({name: round(value, 4) for name, value in example_metrics.items()})

expected = {"Accuracy": 3 / 6, "Precision": 2 / 3, "Recall": 2 / 4, "F1": 4 / 7}
assert all(np.isclose(example_metrics[name], value) for name, value in expected.items()), \
    "A metric does not match the hand calculation"

## Step 9: Compare architectures

The five architectures from Module 5 are retrained with the same three seeds, and the selection rule is unchanged: the architecture with the highest mean validation accuracy is selected, and its seed-20 model is kept as the final network. Validation precision, recall, and F1 are then calculated for each trained network.

In [ ]:
architectures = {
    "1 x 32": dict(n_hidden=1, n_neurons=32),
    "2 x 64": dict(n_hidden=2, n_neurons=64),
    "2 x 64, dropout 0.3": dict(n_hidden=2, n_neurons=64, dropout_rate=0.3),
    "3 x 64, dropout 0.2": dict(n_hidden=3, n_neurons=64, dropout_rate=0.2),
    "3 x 128, dropout 0.3": dict(n_hidden=3, n_neurons=128, dropout_rate=0.3)
}
seeds = [20, 21, 22]

run_rows = []
trained_models = {}

for name, settings in architectures.items():
    for seed in seeds:
        model = build_model(**settings, seed=seed)
        early_stopping_cb = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True)

        start = time.perf_counter()
        run_history = model.fit(
            X_train_nn_prepared, y_train_nn,
            epochs=100,
            validation_data=(X_valid_prepared, y_valid),
            callbacks=[early_stopping_cb],
            verbose=0)
        seconds = time.perf_counter() - start

        train_accuracy = model.evaluate(X_train_nn_prepared, y_train_nn, verbose=0)[1]
        valid_loss, valid_accuracy = model.evaluate(X_valid_prepared, y_valid, verbose=0)

        run_rows.append({
            "Architecture": name,
            "Seed": seed,
            "Parameters": model.count_params(),
            "Epochs run": len(run_history.history["loss"]),
            "Best epoch": int(np.argmin(run_history.history["val_loss"])) + 1,
            "Training accuracy": train_accuracy,
            "Validation accuracy": valid_accuracy,
            "Validation loss": valid_loss,
            "Seconds": seconds
        })
        trained_models[(name, seed)] = (model, run_history)
        print(f"{name}, seed {seed}: validation accuracy {valid_accuracy:.4f}")

run_results = pd.DataFrame(run_rows)

In [ ]:
valid_metric_rows = []

for (name, seed), (model, run_history) in trained_models.items():
    # Probability of default for each validation client, flattened to one value per client
    valid_probabilities = model.predict(X_valid_prepared, verbose=0).ravel()
    # 0/1 predictions using the 0.5 cut-off
    valid_predictions = (valid_probabilities >= 0.5).astype(int)
    # Accuracy, precision, recall and F1 from classification_metrics()
    valid_metrics = classification_metrics(y_valid, valid_predictions)

    valid_metric_rows.append({
        "Architecture": name,
        "Seed": seed,
        "Validation precision": valid_metrics["Precision"],
        "Validation recall": valid_metrics["Recall"],
        "Validation F1": valid_metrics["F1"]
    })

# Add the new columns to the Module 5 results for each run
nn_validation_results = run_results.merge(pd.DataFrame(valid_metric_rows), on=["Architecture", "Seed"])

nn_validation_results.round(4)

Means across the three seeds for each architecture (compare with Table 2 of the midterm).

In [ ]:
architecture_results = nn_validation_results.groupby("Architecture", sort=False).agg(
    Parameters=("Parameters", "first"),
    Mean_epochs=("Epochs run", "mean"),
    Training_accuracy=("Training accuracy", "mean"),
    Validation_accuracy=("Validation accuracy", "mean"),
    Validation_accuracy_std=("Validation accuracy", "std"),
    Validation_precision=("Validation precision", "mean"),
    Validation_recall=("Validation recall", "mean"),
    Validation_F1=("Validation F1", "mean"),
    Validation_F1_std=("Validation F1", "std"),
    Validation_loss=("Validation loss", "mean"),
    Seconds=("Seconds", "mean")
)

architecture_results.round(4)

In [ ]:
# Same selection rule as Module 5: highest mean validation accuracy, then the seed-20 model
best_architecture = architecture_results["Validation_accuracy"].idxmax()
nn_model, nn_history = trained_models[(best_architecture, 20)]

print("Selected architecture:", best_architecture)

In [ ]:
# The architecture with the highest mean validation F1
best_f1_architecture = architecture_results["Validation_F1"].idxmax()

print("Highest mean validation F1:", best_f1_architecture)

## Step 10: Cross-validate the selected network

In Part A, the statistical models were scored with five-fold cross-validation, but in Step 9 the network was scored on a single validation set, so their validation results cannot be compared directly. Here the selected architecture is trained and scored on the same five folds as Part A, using seed 20 as for the final network.

Within each fold, 20% of the training clients are held out for early stopping, so the clients used to score the network play no part in deciding when its training stops. The network therefore learns from 15,360 clients per fold, compared with 19,200 for the statistical models. The scaler is refitted in each fold on the clients the network learns from. The encoder keeps the categories found in Step 6: a payment status of 8 (eight months late), which appears only once or twice among the 24,000 training clients, is missing from some folds, and the network needs the same 89 inputs in every fold.

In [ ]:
# The categories the network's encoder found in Step 6 (89 inputs in total)
nn_categories = nn_preprocessor.named_transformers_["categorical"].categories_
nn_label = f"Neural network ({best_architecture})"

nn_fold_rows = []

# The same five folds as Part A
for fold, (train_index, valid_index) in enumerate(cv.split(X_train, y_train), start=1):
    X_fold_train, y_fold_train = X_train.iloc[train_index], y_train.iloc[train_index]
    X_fold_valid, y_fold_valid = X_train.iloc[valid_index], y_train.iloc[valid_index]

    # Hold out 20% of the fold's training clients for early stopping, as in Step 6
    X_fold_fit, X_fold_stop, y_fold_fit, y_fold_stop = train_test_split(
        X_fold_train,
        y_fold_train,
        test_size=0.20,
        random_state=20,
        stratify=y_fold_train
    )

    # The scaler is fitted on this fold's network training clients only
    fold_preprocessor = clone(nn_preprocessor).set_params(categorical__categories=nn_categories)
    X_fold_fit_prepared = fold_preprocessor.fit_transform(X_fold_fit)
    X_fold_stop_prepared = fold_preprocessor.transform(X_fold_stop)
    X_fold_valid_prepared = fold_preprocessor.transform(X_fold_valid)

    model = build_model(**architectures[best_architecture], seed=20)
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True)

    start = time.perf_counter()
    fold_history = model.fit(
        X_fold_fit_prepared, y_fold_fit,
        epochs=100,
        validation_data=(X_fold_stop_prepared, y_fold_stop),
        callbacks=[early_stopping_cb],
        verbose=0)
    seconds = time.perf_counter() - start

    # 0/1 predictions with the same 0.5 cut-off as before
    fit_predictions = (model.predict(X_fold_fit_prepared, verbose=0).ravel() >= 0.5).astype(int)
    fold_predictions = (model.predict(X_fold_valid_prepared, verbose=0).ravel() >= 0.5).astype(int)
    train_metrics = classification_metrics(y_fold_fit, fit_predictions)
    fold_metrics = classification_metrics(y_fold_valid, fold_predictions)

    nn_fold_rows.append({
        "Fold": fold,
        "Training accuracy": train_metrics["Accuracy"],
        "Validation accuracy": fold_metrics["Accuracy"],
        "Validation precision": fold_metrics["Precision"],
        "Validation recall": fold_metrics["Recall"],
        "Training F1": train_metrics["F1"],
        "Validation F1": fold_metrics["F1"],
        "Epochs run": len(fold_history.history["loss"]),
        "Best epoch": int(np.argmin(fold_history.history["val_loss"])) + 1,
        "Seconds": seconds
    })
    print(f"Fold {fold}: validation accuracy {fold_metrics['Accuracy']:.4f}, F1 {fold_metrics['F1']:.4f}")

nn_cv_folds = pd.DataFrame(nn_fold_rows).set_index("Fold")

nn_cv_folds.round(4)

Means across the five folds, with the network added to the Part A results. The F1 gap is training F1 minus validation F1. The network's training scores are measured on the 15,360 clients it learned from in each fold.

In [ ]:
cv_metric_columns = ["Training accuracy", "Validation accuracy", "Validation precision",
                     "Validation recall", "Training F1", "Validation F1"]

all_cv_summary = cv_summary.copy()
all_cv_summary.loc[nn_label] = nn_cv_folds[cv_metric_columns].mean()
all_cv_summary["F1 gap"] = all_cv_summary["Training F1"] - all_cv_summary["Validation F1"]

all_cv_summary.round(4)

Standard deviations across the five folds.

In [ ]:
all_cv_std = cv_std.copy()
all_cv_std.loc[nn_label] = nn_cv_folds[cv_metric_columns].std()

all_cv_std.round(4)

Validation F1 in each fold. Because every model was scored on the same folds, the models can also be compared fold by fold.

In [ ]:
fold_f1 = pd.DataFrame({
    model_name: fold_results["Validation F1"]
    for model_name, fold_results in cv_fold_results.items()
})
fold_f1[nn_label] = nn_cv_folds["Validation F1"]

# One row per model and one column per fold
fold_f1.T.round(4)

# Part C: Test-set comparison of all four models

## Step 11: Refit the statistical models and predict the test set

As in Module 5, the statistical models are refitted on all 24,000 training clients with their Module 3 settings. All four models use a 0.5 probability cut-off to predict default.

In [ ]:
final_models = {
    "Logistic regression": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Decision tree (depth 5)": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", DecisionTreeClassifier(max_depth=5, random_state=20))
    ]),
    "Random forest": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=20, n_jobs=-1))
    ])
}

test_predictions = pd.DataFrame({"Actual": y_test})
test_probabilities = pd.DataFrame(index=y_test.index)
training_seconds = {}

for model_name, model in final_models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    training_seconds[model_name] = time.perf_counter() - start

    test_predictions[model_name] = model.predict(X_test)
    test_probabilities[model_name] = model.predict_proba(X_test)[:, 1]

# The network outputs a probability; 0.5 is the same cut-off predict() uses above
nn_probabilities = nn_model.predict(X_test_prepared, verbose=0).ravel()
test_probabilities["Neural network"] = nn_probabilities
test_predictions["Neural network"] = (nn_probabilities >= 0.5).astype(int)

selected_runs = run_results[run_results["Architecture"] == best_architecture]
training_seconds["Neural network"] = selected_runs.loc[selected_runs["Seed"] == 20, "Seconds"].iloc[0]

## Step 12: Test-set accuracy, precision, recall, F1, and ROC AUC

In [ ]:
test_rows = []

for model_name in test_probabilities.columns:
    predictions = test_predictions[model_name]

    # Accuracy, precision, recall and F1 on the test set
    test_metrics = classification_metrics(y_test, predictions)

    test_rows.append({
        "Model": model_name,
        **test_metrics,
        "ROC AUC": roc_auc_score(y_test, test_probabilities[model_name]),
        "Training seconds": training_seconds[model_name]
    })

test_results = pd.DataFrame(test_rows).set_index("Model")

test_results.round(4)

## Step 13: Confusion matrices

These are the counts behind the metrics above. Precision uses the "Default" column of predictions, recall uses the "Default" row of actual outcomes, and F1 combines the two. None of the three uses the correctly predicted non-defaults, which make up most of the test set, so unlike accuracy they are not inflated by the majority class.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, model_name in zip(axes, test_results.index):
    ConfusionMatrixDisplay.from_predictions(
        test_predictions["Actual"],
        test_predictions[model_name],
        labels=[0, 1],
        display_labels=["No default", "Default"],
        cmap="Blues",
        colorbar=False,
        values_format="d",
        im_kw={"vmin": 0, "vmax": len(y_test)},
        ax=ax
    )

    ax.set_title(model_name)
    ax.set_xlabel("Predicted outcome")
    ax.set_ylabel("Actual outcome")

plt.tight_layout()
plt.show()

## Step 14: Results for each class

The precision, recall, and F1 values above describe the default class only. `classification_report()` gives the same metrics for both classes, along with their macro average (both classes weighted equally) and weighted average (each class weighted by its number of clients).

In [ ]:
for model_name in test_probabilities.columns:
    print(model_name)
    # The classification report for this model's test predictions
    print(classification_report(y_test, test_predictions[model_name],
                                target_names=["No default", "Default"], digits=4))

## Step 15: How each metric ranks the models

Rank 1 is the model with the highest score on that metric.

In [ ]:
metric_columns = ["Accuracy", "Precision", "Recall", "F1", "ROC AUC"]

# Rank the models on each metric (1 = highest score)
test_rankings = test_results[metric_columns].rank(ascending=False)

test_rankings

# Part D: Hyperparameter tuning of logistic regression

## Step 16: Grid search

Logistic regression is tuned because it was the most consistent of the four models: it had the highest cross-validated accuracy and the highest test F1, its cross-validated F1 was within the fold-to-fold variation of the best, it barely overfitted, and its coefficients can be interpreted. It also trains quickly, so many settings can be compared.

The settings are selected on F1, because accuracy rewards predicting the majority class: predicting no default for every client is 77.9% accurate. Precision, recall, and accuracy are recorded alongside it.

The grid search (Géron, 2022) tries 24 combinations of three hyperparameters:

- `C`, the inverse of the regularization strength, from 0.001 to 100. Smaller values penalize large coefficients more.
- `class_weight`: none, or `"balanced"`, which gives each default about 3.5 times the weight of a non-default during training.
- The penalty: L2, which shrinks all coefficients (default `lbfgs` solver), or L1, which can set coefficients to exactly zero (`liblinear` solver, because `lbfgs` does not support L1). This version of scikit-learn sets the penalty with `l1_ratio` (0 for L2, 1 for L1).

The untuned settings (`C` = 1, no class weights, L2) are one of the combinations, so they are scored on the same five folds as Part A. The combinations are trained one at a time so that their training times are comparable, and the solver's iteration count is recorded for each fold as a measure of convergence.

In [ ]:
from sklearn.model_selection import GridSearchCV, TunedThresholdClassifierCV
from sklearn.metrics import confusion_matrix, make_scorer
from scipy.stats import bootstrap


def solver_iterations(estimator, X, y):
    """Not a score: the number of iterations the solver needed on this fold, recorded to compare convergence."""
    return estimator.named_steps["model"].n_iter_[0]


C_values = [0.001, 0.01, 0.1, 1, 10, 100]

param_grid = [
    {   # L2 penalty with the default lbfgs solver
        "model__l1_ratio": [0],
        "model__solver": ["lbfgs"],
        "model__C": C_values,
        "model__class_weight": [None, "balanced"]
    },
    {   # L1 penalty, which needs the liblinear solver
        "model__l1_ratio": [1],
        "model__solver": ["liblinear"],
        "model__C": C_values,
        "model__class_weight": [None, "balanced"]
    }
]

tuning_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),
    # random_state only affects liblinear, which shuffles the data; lbfgs ignores it
    ("model", LogisticRegression(max_iter=1000, random_state=20))
])

grid_search = GridSearchCV(
    tuning_pipeline,
    param_grid,
    cv=cv,  # the same five folds as Part A
    scoring={
        "F1": "f1",
        # 0 rather than a warning if a combination predicts no defaults at all
        "Precision": make_scorer(precision_score, zero_division=0),
        "Recall": "recall",
        "Accuracy": "accuracy",
        "Iterations": solver_iterations
    },
    refit="F1",  # select on F1, then refit the best combination on all 24,000 training clients
    error_score="raise"
)

start = time.perf_counter()
grid_search.fit(X_train, y_train)
search_seconds = time.perf_counter() - start

print(f"{len(grid_search.cv_results_['params'])} combinations x 5 folds: {search_seconds:.0f} seconds")

## Step 17: Results for every combination

Means across the five folds, ranked by F1. F1 SD is the standard deviation across the folds, as in Part A. Training time is per fit and includes the preprocessing. The two solvers count iterations differently, so iterations should be compared within a penalty rather than between L1 and L2.

With L1 and the strongest regularization (`C` = 0.001), the penalty removes all but two of the 89 features and the model predicts no defaults in any fold, so its accuracy equals the 77.88% baseline and its precision, recall, and F1 are zero.

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)

tuning_results = pd.DataFrame({
    "Penalty": results["param_model__l1_ratio"].map({0: "L2", 1: "L1"}),
    "Class weight": results["param_model__class_weight"].fillna("none"),
    "C": results["param_model__C"].astype(float),
    "F1": results["mean_test_F1"],
    # Sample standard deviation across the folds, as in Part A
    "F1 SD": results[[f"split{fold}_test_F1" for fold in range(5)]].std(axis=1),
    "Precision": results["mean_test_Precision"],
    "Recall": results["mean_test_Recall"],
    "Accuracy": results["mean_test_Accuracy"],
    "Training seconds": results["mean_fit_time"],
    "Iterations": results["mean_test_Iterations"]
})

tuning_results = tuning_results.sort_values("F1", ascending=False).reset_index(drop=True)
tuning_results.index = tuning_results.index + 1
tuning_results.index.name = "Rank"

tuning_results.round(4)

The same results plotted against `C`. Color shows the class weights and line style the penalty.

In [ ]:
weight_colors = {"none": "#2a78d6", "balanced": "#eb6834"}
penalty_styles = {"L2": dict(linestyle="-", marker="o"), "L1": dict(linestyle="--", marker="s")}
panels = [("F1", "Validation F1"),
          ("Training seconds", "Training time per fit (s)"),
          ("Iterations", "Solver iterations")]

fig, axes = plt.subplots(3, 1, figsize=(6.5, 7), sharex=True)

for ax, (column, label) in zip(axes, panels):
    for penalty, style in penalty_styles.items():
        for weight, color in weight_colors.items():
            rows = tuning_results[(tuning_results["Penalty"] == penalty)
                                  & (tuning_results["Class weight"] == weight)].sort_values("C")
            weight_label = "balanced class weights" if weight == "balanced" else "no class weights"
            ax.plot(rows["C"], rows[column], color=color, linewidth=1.6, markersize=6,
                    markeredgecolor="white", markeredgewidth=0.8,
                    label=f"{penalty}, {weight_label}", **style)
    ax.set_xscale("log")
    ax.minorticks_off()
    ax.set_ylabel(label, fontsize=10)
    ax.grid(True, color="#e6e6e6", linewidth=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=9)

axes[-1].set_xticks(C_values, labels=[f"{c:g}" for c in C_values])
axes[-1].set_xlabel("C (inverse of regularization strength, log scale)", fontsize=10)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## Step 18: The selected settings

The best combination is compared with the untuned settings from the same grid. Its mean F1 is the highest of 24 averages that each contain some fold-to-fold noise, so it probably overstates how well the settings will do on new data. Step 19 evaluates the reused exploratory holdout; because prior test results influenced model selection, this is not an unbiased final check.

In [ ]:
print("Selected settings:", grid_search.best_params_)

untuned_rank = tuning_results.index[(tuning_results["Penalty"] == "L2")
                                    & (tuning_results["Class weight"] == "none")
                                    & (tuning_results["C"] == 1)][0]

selected_vs_untuned = tuning_results.loc[[1, untuned_rank]]
selected_vs_untuned.index = ["Selected (rank 1)", f"Untuned (rank {untuned_rank})"]

selected_vs_untuned.round(4)

## Step 19: Test-set evaluation

The selected model, already refitted on all 24,000 training clients, is scored on the previously examined exploratory holdout and compared with the untuned model from Part C. A second way of trading precision for recall is included for comparison: the untuned model with its 0.5 cut-off replaced by the cut-off that maximizes F1 across the same five folds (`TunedThresholdClassifierCV`). The counts show how many of the 1,327 defaulters in the test set each model caught, and how many non-defaulters it wrongly flagged.

In [ ]:
# The untuned model, with its cut-off chosen on the five folds instead of fixed at 0.5
threshold_model = TunedThresholdClassifierCV(
    Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    scoring="f1",
    cv=cv
)
threshold_model.fit(X_train, y_train)
print(f"Cut-off that maximizes F1 across the folds: {threshold_model.best_threshold_:.3f}")

tuned_model = grid_search.best_estimator_
print(f"Training time on all 24,000 clients: untuned {training_seconds['Logistic regression']:.2f} s, "
      f"tuned {grid_search.refit_time_:.2f} s (the whole grid search took {search_seconds:.0f} s)")

tuning_test_predictions = {
    "Untuned (Part C)": test_predictions["Logistic regression"],
    "Tuned (grid search)": tuned_model.predict(X_test),
    "Untuned, tuned cut-off": threshold_model.predict(X_test)
}
tuning_test_probabilities = {
    "Untuned (Part C)": test_probabilities["Logistic regression"],
    "Tuned (grid search)": tuned_model.predict_proba(X_test)[:, 1],
    "Untuned, tuned cut-off": threshold_model.predict_proba(X_test)[:, 1]
}

tuning_test_rows = []
for name, predictions in tuning_test_predictions.items():
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    tuning_test_rows.append({
        "Model": name,
        **classification_metrics(y_test, predictions),
        "ROC AUC": roc_auc_score(y_test, tuning_test_probabilities[name]),
        "Defaults caught": tp,
        "Defaults missed": fn,
        "False alarms": fp
    })

tuning_test_results = pd.DataFrame(tuning_test_rows).set_index("Model")

tuning_test_results.round(4)

Paired bootstrap 95% intervals for the change in exploratory-holdout F1 compared with the untuned model, using SciPy's `bootstrap()` as in Géron (2022). The 6,000 holdout clients are resampled with replacement 2,000 times, and both fixed fitted models are scored on each resample. These intervals describe resampling uncertainty conditional on those fitted models and this reused holdout. They do not account for model selection, threshold selection or training variability, correct selection bias, or establish performance on new populations. An interval excluding zero applies only within this conditional comparison.

In [ ]:
y_test_values = y_test.to_numpy()
untuned_test_predictions = np.asarray(tuning_test_predictions["Untuned (Part C)"])


def f1_gain(y_true, new_predictions, old_predictions):
    return f1_score(y_true, new_predictions) - f1_score(y_true, old_predictions)


gain_rows = []
for name in ["Tuned (grid search)", "Untuned, tuned cut-off"]:
    new_predictions = np.asarray(tuning_test_predictions[name])
    interval = bootstrap(
        (y_test_values, new_predictions, untuned_test_predictions),
        f1_gain,
        paired=True,
        vectorized=False,
        n_resamples=2000,
        method="percentile",
        rng=np.random.default_rng(20)
    ).confidence_interval
    gain_rows.append({
        "Model": name,
        "Test F1 gain": f1_gain(y_test_values, new_predictions, untuned_test_predictions),
        "95% CI lower": interval.low,
        "95% CI upper": interval.high
    })

pd.DataFrame(gain_rows).set_index("Model").round(4)

## Step 20: What drives the tuned model's predictions

Positive coefficients raise the predicted probability of default and negative ones lower it. A rare category can have a large coefficient yet affect few clients, so the features are ranked by the coefficient multiplied by the feature's standard deviation in the training data, which reflects how much each feature moves the predictions across clients. The numerical features were standardized, so their standard deviation is 1.

In the feature names, `PAY_0` is the repayment status in September 2005, the most recent month: values of 1 or more count the months a payment was late, and −1 means paid on time (Yeh, 2009); −2 and 0 also occur but are not defined in the dataset's documentation. For example, `PAY_0_2` is 1 for clients whose September payment was two months late.

In [ ]:
preprocessing = tuned_model.named_steps["preprocessing"]
feature_names = [name.split("__", 1)[1] for name in preprocessing.get_feature_names_out()]

X_train_transformed = preprocessing.transform(X_train)
if hasattr(X_train_transformed, "toarray"):  # the one-hot encoder can return a sparse matrix
    X_train_transformed = X_train_transformed.toarray()

coefficient_table = pd.DataFrame({
    "Coefficient": tuned_model.named_steps["model"].coef_[0],
    "Feature SD": X_train_transformed.std(axis=0)
}, index=feature_names)
coefficient_table["Coefficient x SD"] = coefficient_table["Coefficient"] * coefficient_table["Feature SD"]

# The ten features that move the predictions most across clients
coefficient_table.reindex(
    coefficient_table["Coefficient x SD"].abs().sort_values(ascending=False).index
).head(10).round(3)

## Attribution and disclosure

Dataset: Yeh (2009), UCI Default of Credit Card Clients, https://doi.org/10.24432/C55S3H, CC BY 4.0. Methods cite Geron (2022), Hands-On Machine Learning, 3rd ed. See README for complete references.

The original Module 3 disclosure describes AI assistance for code, explanations, interpretation, editing and portions of writing, with Benjamin reviewing the results. This portfolio adaptation also used AI assistance. Benjamin confirms no instructor code was used in this project and all sources used are referenced. The separate reinforcement-learning assignment contains instructor code and is excluded.
